In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

llm = init_chat_model(model="gpt-4o-mini")

In [ ]:
load_dotenv(override=True)

In [ ]:

class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
@tool
def get_stock_price(ticker: str) -> str:
    """
    Return the current price of a stock or ticker requested by the user/customer.
    """
    print(ticker)
    return {
        "TATASTEEL.NS": 232.45,
        "INFY": 189.85
    }.get(ticker, 0.0)

tools = [get_stock_price]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def chatbot(state: State) -> State:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)

builder.add_node(chatbot)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("chatbot", END)

graph = builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
message = {"role": "user", "content": "What is the current price of Tata Steel, as of today?"}
response = graph.invoke({"messages": [message]})
print(response['messages'][-1].content)